# Predicting Heat Capacity ($C_p$) with a Neural Network — step by step

**Materials Informatics Workshop — Day 3 (Deep Learning part)**

This notebook continues the *classical Machine Learning* part (which you already did).
Now we solve **the same problem** — predicting the heat capacity $C_p$ of inorganic
materials from chemical composition and temperature — but using a **neural network**
(deep learning) instead of the `scikit-learn` models.

> Based on: Wang, Murdock, Kauwe, Oliynyk, Gurlo, Brgoch, Persson, Sparks,
> *"Machine Learning for Materials Scientists: An Introductory Guide toward Best Practices"*,
> **Chem. Mater.** 2020, 32 (12), 4954–4965. DOI: 10.1021/acs.chemmater.0c01907
> ([GitHub repository](https://github.com/anthony-wang/BestPractices)).

### The idea
>Think of a neural network as a **flexible machine** that turns numbers into numbers. You feed it information about a material (its composition and temperature), and it gives back a guess for the heat capacity $C_p$. Inside, the machine has many small **knobs** (called weights). Turning these knobs changes the guess.
>At the start, the knobs are random, so the guesses are bad. **Training** fixes this. We show the machine an example where we already know the real $C_p$. It makes a guess, we check how wrong it is, and we turn the knobs a little to make the guess better. We do this many times, with many examples. Slowly, the machine learns to guess well.

### What we will do (notebook map)
1. Import the libraries
2. Load the data (already split into train/validation/test)
3. Turn chemical formulas into numbers (**CBFV featurization**)
4. Put the numbers on the same scale (**scaling + normalization**)
5. Understand and **build the neural network** (`DenseNet`)
6. Choose the **device** (CPU or GPU)
7. Prepare the **DataLoader** (serve the data in batches)
8. Define the **loss function** and **optimizer**
9. Scale the **target** ($C_p$)
10. **Train** the network (the heart of the notebook)
11. Evaluate on **validation** + loss curve
12. Evaluate on **test** (only once!)
13. **Save and load** the model

Each step has a short explanation before the code, and the code is full of comments.
Run it cell by cell (Shift+Enter), no rush.


## Step 1 — Import the libraries

Each `import` brings a "toolbox". Here is what each one does:

- **os / numpy / pandas / matplotlib** → handle paths, numbers, tables and plots.
- **CBFV** → turns the chemical formula (text, e.g. `"Al2O3"`) into a vector of numbers.
- **sklearn.preprocessing** → `StandardScaler` and `normalize` adjust the scale of the numbers.
- **sklearn.metrics** → metrics to measure how good the model is ($R^2$, MAE, RMSE).
- **torch** → PyTorch, the deep learning library. `nn` has the network layers,
  `optim` has the optimizers, and `Dataset`/`DataLoader` serve the data in batches.

The last line fixes a **random seed** (`RNG_SEED = 42`). This makes the result
**reproducible**: running again, you get the same numbers.


In [1]:
import os
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Chemical-composition featurization
import sys
sys.path.append(os.path.abspath('../cbfv/cbfv'))
from composition import generate_features

# Scaling/normalization and metrics
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# PyTorch (deep learning)
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

# Random seed -> reproducible results
RNG_SEED = 42
np.random.seed(RNG_SEED)
torch.manual_seed(RNG_SEED)

print('Libraries imported successfully!')
print('PyTorch version:', torch.__version__)

Libraries imported successfully!
PyTorch version: 1.13.1+cu117


## Step 2 — Load the data

We read the three `.csv` files into pandas **DataFrames** (tables).
We do not touch the train/validation/test split — it was already done correctly before
(grouping by formula, so the same compound does **not** appear in two sets).

`.shape` shows `(number of rows, number of columns)`.


In [2]:
PATH = os.getcwd()
train_path = os.path.join(PATH, '../day3/train_formula.csv')
valid_path = os.path.join(PATH, '../day3/valid_formula.csv')
test_path  = os.path.join(PATH, '../day3/test_formula.csv')

df_train   = pd.read_csv(train_path)
df_valid   = pd.read_csv(valid_path)
df_test    = pd.read_csv(test_path)

print(f'Train:      {df_train.shape}')
print(f'Validation: {df_valid.shape}')
print(f'Test:       {df_test.shape}')

# Peek at the first rows of the training set
df_train.head()

Train:      (3309, 4)
Validation: (807, 4)
Test:       (448, 4)


,Unnamed: 0,FORMULA,T (K),Cp (J/molK)
0,4209,O2Ti1,1500.0,76.944
1,573,Be1F4Li2,800.0,210.100
2,2494,N0.465V1,300.0,28.971
3,3654,I1K1,700.0,59.777
4,4538,W1,1500.0,29.862


## Step 3 — Turn formulas into numbers (CBFV featurization)

The neural network does not understand `"Al2O3"`. It only understands **numbers**. We need to
convert each formula into a vector of numbers — this is **featurization**.

We use **CBFV** (*Composition-Based Feature Vector*). The idea is simple: for each element
in the formula, CBFV takes known properties (atomic number, radius, electronegativity, etc.)
and combines them (sum, mean, max…) to describe the whole compound as **one vector**.

Important detail: `generate_features` expects a column named **`target`**.
In our case the target is `Cp`, so we rename `Cp → target` before calling the function.

The arguments:
- `elem_prop='oliynyk'` → which table of element properties to use.
- `drop_duplicates=False` → keep repeated formulas (we have the same compound at several temperatures).
- `extend_features=True` → include extra columns (the temperature `T`).
- `sum_feat=True` → include the sum *features*.

The function returns four things: the feature matrix `X`, the target `y`, the formulas, and the skipped ones.


In [3]:
rename_dict = {'FORMULA':'formula','T (K)':'T',
               'Cp (J/molK)':'target'}
cp_train_sample = df_train.rename(columns=rename_dict)
cp_valid_sample = df_valid.rename(columns=rename_dict)
cp_test_sample = df_test.rename(columns=rename_dict)

In [4]:
X_train_unscaled, y_train, formula_train, skipped_train = generate_features(
    cp_train_sample,
    elem_prop='oliynyk', 
    drop_duplicates=False, 
    extend_features=True, 
    sum_feat=True)

X_valid_unscaled, y_valid, formula_valid, skipped_valid = generate_features(
    cp_valid_sample, 
    elem_prop='oliynyk', 
    drop_duplicates=False, 
    extend_features=True, 
    sum_feat=True)

X_test_unscaled, y_test, formula_test, skipped_test = generate_features(
    cp_test_sample, 
    elem_prop='oliynyk', 
    drop_duplicates=False, 
    extend_features=True, 
    sum_feat=True)

Processing Input Data: 100%|█████████████| 3309/3309 [00:00<00:00, 15588.95it/s]


	Featurizing Compositions...


Assigning Features...: 100%|██████████████| 3309/3309 [00:00<00:00, 8989.28it/s]


	Creating Pandas Objects...


Processing Input Data: 100%|████████████████| 807/807 [00:00<00:00, 9007.93it/s]


...

	Creating Pandas Objects...


Processing Input Data: 100%|███████████████| 448/448 [00:00<00:00, 10519.57it/s]


	Featurizing Compositions...


Assigning Features...: 100%|████████████████| 448/448 [00:00<00:00, 4782.66it/s]


	Creating Pandas Objects...


Note the number of **columns** above (it should be **177** with the `oliynyk` featurizer).
This number will be the **input dimension** of our neural network later on. Keep that in mind.


## Step 4 — Put everything on the same scale

The columns have very different magnitudes (an atomic number goes up to ~100, a temperature
up to thousands of K). Neural networks train **much better** when the features are on similar
scales. We do this in two steps:

1. **`StandardScaler`** (acts per *column*): makes each column have **mean 0 and std 1**.
2. **`normalize`** (acts per *row*): makes each row have **norm 1**.

⚠️ **Golden rule against data leakage:**
the scaler learns the mean and the std **from the training set only** (`.fit_transform`).
For validation and test, we **only apply** what was learned (`.transform`).
This way test information never "leaks" into training.


In [5]:
# 1) Per-column standardization: fit ONLY on the training set
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_unscaled)   # learns AND applies on the training set
X_valid   = scaler.transform(X_valid_unscaled)         # ONLY applies (uses training statistics)
X_test  = scaler.transform(X_test_unscaled)        # ONLY applies

# 2) Per-row normalization (norm 1 for each sample)
#
X_train = normalize(X_train)
X_valid   = normalize(X_valid)
X_test  = normalize(X_test)

print('Done. X_train now has mean ~0 per column and each row with norm 1.')
print('Final shape of X_train:', X_train.shape)
print('Mean of all X_train values (just to peek):', X_train.mean())

Done. X_train now has mean ~0 per column and each row with norm 1.
Final shape of X_train: (3309, 178)
Mean of all X_train values (just to peek): 0.0008996604661410418


## Step 5 — Understand and build the neural network

### What is a dense neural network, really?
Imagine a queue of "layers". Each layer does two things to the numbers it receives:
1. a **linear combination** (multiply by weights and sum) — `nn.Linear`;
2. an **activation function** that bends the result, giving flexibility — here `LeakyReLU`.

By stacking layers, the network becomes a function able to learn complicated relationships
between the material's composition and $C_p$.

Our network (`DenseNet`):
- **Input:** 177 numbers (the CBFV features).
- **Hidden layer(s):** where the "magic" happens (we will use 1 layer with 16 neurons).
- **Output:** 1 number → the predicted $C_p$.

The `dropout` (optional) randomly "turns off" neurons during training to avoid
**overfitting** (memorizing the training set). We will leave it at 0 for simplicity.

You don't need to understand every line of the class now — what matters is the structure
**input → hidden layers → output**.


In [6]:
class DenseNet(nn.Module):
    """A simple fully-connected neural network.

    input_dims  : number of input features (177 in our case)
    hidden_dims : a list with the size of each hidden layer, e.g. [16] or [64, 32]
    output_dims : number of outputs (1, because we predict only Cp)
    dropout     : fraction of neurons turned off during training (0.0 = none)
    """
    def __init__(self, input_dims, hidden_dims=[16], output_dims=1, dropout=0.0):
        super().__init__()

        # Build a plain list of layers, one "block" per hidden layer.
        # A block is: Linear (combine) -> Dropout (optional) -> LeakyReLU (activation).
        layers = []
        in_size = input_dims
        for h in hidden_dims:
            layers.append(nn.Linear(in_size, h))   # linear combination
            layers.append(nn.Dropout(dropout))     # optional: turn off some neurons
            layers.append(nn.LeakyReLU())          # activation: bends the result
            in_size = h                            # next layer starts where this one ended

        # Output layer: from the last hidden size to the single Cp value
        layers.append(nn.Linear(in_size, output_dims))

        # nn.ModuleList = a normal Python list that PyTorch knows how to track
        self.network = nn.ModuleList(layers)

    def forward(self, x):
        # Pass the data through each layer, one after another
        for layer in self.network:
            x = layer(x)
        return x

print('DenseNet class defined. (We have not created the model yet - only the "recipe".)')

DenseNet class defined. (We have not created the model yet - only the "recipe".)


## Step 6 — Choose the device (CPU or GPU)

Neural networks train faster on a **GPU** (graphics card). PyTorch automatically detects
whether there is a CUDA-compatible GPU. If not, it uses the **CPU** (works the same,
just slower). Our model is small, so it runs fine on either.


In [7]:
if torch.cuda.is_available():
    compute_device = torch.device('cuda')
else:
    compute_device = torch.device('cpu')

print('GPU (CUDA) available?', torch.cuda.is_available())
print('Chosen device:', compute_device)

GPU (CUDA) available? False
Chosen device: cpu


## Step 7 — Serve the data in batches (Dataset + DataLoader)

Instead of throwing **all** the data into the network at once, we train in **batches** —
small chunks. This is more efficient and helps training.

- The **`Dataset`** knows **how many** examples there are (`__len__`) and how to **get one**
  example by index (`__getitem__`).
- The **`DataLoader`** takes that Dataset and serves it in batches, shuffling when useful.

Let's define a simple version of these two for our `X` and `y` arrays.


In [8]:
class CBFVDataset(Dataset):
    '''Wraps (X, y) so PyTorch can iterate over it.'''
    def __init__(self, dataset):
        self.X = np.array(dataset[0])
        self.y = np.array(dataset[1])

    def __len__(self):
        # How many examples we have
        return self.X.shape[0]

    def __getitem__(self, idx):
        # How to get ONE example (row idx), already as PyTorch tensors
        X = torch.as_tensor(self.X[[idx], :])
        y = torch.as_tensor(np.array(self.y[idx]))
        return (X, y)


# Wrap each set
train_dataset = CBFVDataset((X_train, y_train))
valid_dataset   = CBFVDataset((X_valid,   y_valid))
test_dataset  = CBFVDataset((X_test,  y_test))

# Create the DataLoaders (split the data into batches of `batch_size`)
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(valid_dataset,   batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

print(f'Batches of {batch_size} examples ready.')
print('Train:', len(train_dataset), 'examples |',
      'Validation:', len(valid_dataset), '|', 'Test:', len(test_dataset))

Batches of 64 examples ready.
Train: 3309 examples | Validation: 807 | Test: 448


## Step 8 — Create the model, the loss function and the optimizer

Now we finally **create** the model from the `DenseNet` recipe.
The input dimension is the number of feature columns (177).

- **Loss function (`criterion`)**: measures the error between the prediction and the real value.
  We use `L1Loss` (= mean absolute error, MAE) — easy to interpret.
- **Optimizer (`optimizer`)**: the one that adjusts the network weights to reduce the loss.
  We use **Adam**, a very good default choice. The `lr` (*learning rate*) controls
  the step size at each adjustment.

Finally, we move the model and the loss to the chosen device (CPU/GPU).


In [9]:
# Input dimension = number of features (columns) of X
input_dims = X_train.shape[1]
print('Input dimension (number of features):', input_dims)

# Create the model: 1 hidden layer with 16 neurons
model = DenseNet(input_dims, hidden_dims=[16], dropout=0.0)

# Loss function: mean absolute error (MAE)
criterion = nn.L1Loss()

# Optimizer: Adam, with learning rate 0.001
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Move model and loss to the device (CPU or GPU)
model = model.to(compute_device)
criterion = criterion.to(compute_device)

print('\nModel structure:')
print(model)

Input dimension (number of features): 178

Model structure:
DenseNet(
  (network): ModuleList(
    (0): Linear(in_features=178, out_features=16, bias=True)
    (1): Dropout(p=0.0, inplace=False)
    (2): LeakyReLU(negative_slope=0.01)
    (3): Linear(in_features=16, out_features=1, bias=True)
  )
)


## Step 9 — Scale the target ($C_p$)

Just as we scaled the *features*, it helps to scale the **target** ($C_p$). Here we use a trick:
take the **logarithm** of the target and then standardize (mean 0, std 1). This usually stabilizes
training when the values vary a lot.

We create a small class, `MeanLogNormScaler`, that knows how to:
- **`scale`** → transform $C_p$ (real) → internal scale (for the network to learn);
- **`unscale`** → transform the network's prediction back to J/mol·K (for us to read).


In [10]:
class MeanLogNormScaler:
    '''Scales the target via log + standardization; and undoes the transformation.'''
    def __init__(self, data):
        data = torch.as_tensor(data)
        self.logdata = torch.log(data)
        self.mean = torch.mean(self.logdata)
        self.std  = torch.std(self.logdata)

    def scale(self, data):
        data = torch.as_tensor(data)
        return (torch.log(data) - self.mean) / self.std

    def unscale(self, data_scaled):
        data_scaled = torch.as_tensor(data_scaled) * self.std + self.mean
        return torch.exp(data_scaled)

    def state_dict(self):
        return {'mean': self.mean, 'std': self.std}

    def load_state_dict(self, state_dict):
        self.mean = state_dict['mean']
        self.std  = state_dict['std']


# Fit the target scaler using the TRAINING y
target_scaler = MeanLogNormScaler(train_dataset.y)
print('Target scaler ready (mean={:.3f}, std={:.3f}).'.format(
    float(target_scaler.mean), float(target_scaler.std)))

Target scaler ready (mean=4.478, std=0.578).


## Step 10 — Helper functions: predict and evaluate

Before training, we define two helpers:

- **`predict`**: passes the data through the network and returns lists of actual and predicted values
  (already "unscaling" the target back to J/mol·K).
- **`evaluate`**: computes three metrics from those values:
  - **$R^2$** (higher is better; 1.0 is perfect),
  - **MAE** (mean absolute error; lower is better),
  - **RMSE** (root mean squared error; penalizes large errors).

And a plotting function **`plot_pred_act`**: predicted × actual. Points on the dashed
line = perfect prediction.


In [11]:
data_type = torch.float

def predict(model, data_loader):
    '''Runs the model in evaluation mode and returns (actual_values, predicted_values).'''
    target_list, pred_list = [], []
    model.eval()  # evaluation mode (turns off dropout etc.)
    with torch.no_grad():  # no need to compute gradients here
        for X, y_act in data_loader:
            X = X.to(compute_device, dtype=data_type)
            y_act = y_act.cpu().flatten().tolist()
            y_pred = model.forward(X).cpu().flatten().tolist()
            # Undo the target scaling -> back to J/mol K
            y_pred = target_scaler.unscale(y_pred).tolist()
            target_list.extend(y_act)
            pred_list.extend(y_pred)
    model.train()  # back to training mode
    return target_list, pred_list


def evaluate(target, pred):
    '''Computes R2, MAE and RMSE.'''
    r2   = r2_score(target, pred)
    mae  = mean_absolute_error(target, pred)
    rmse = np.sqrt(mean_squared_error(target, pred))  # RMSE robust to sklearn versions
    return r2, mae, rmse


def plot_pred_act(act, pred, model, label='$C_p$ (J / mol K)'):
    xy_max = np.max([np.max(act), np.max(pred)])
    fig = plt.figure(figsize=(6, 6))
    plt.plot(act, pred, 'o', ms=9, mec='k', mfc='silver', alpha=0.4)
    plt.plot([0, xy_max], [0, xy_max], 'k--', label='ideal')
    polyfit = np.polyfit(act, pred, deg=1)
    reg_ys = np.poly1d(polyfit)(np.unique(act))
    plt.plot(np.unique(act), reg_ys, alpha=0.8, label='linear fit')
    plt.axis('scaled')
    plt.xlabel(f'{label} actual')
    plt.ylabel(f'{label} predicted')
    plt.title(f'{type(model).__name__} — R2: {r2_score(act, pred):0.4f}')
    plt.legend(loc='upper left')
    return fig

print('Helper functions ready.')

Helper functions ready.


## Step 11 — Train the network (the heart of the notebook)

Training = repeating many times (each repetition is an **epoch**) the following cycle,
batch by batch:

1. **Scale** the batch target;
2. Send the data to the device;
3. **Zero** the old gradients (`optimizer.zero_grad()`);
4. **Forward**: the network makes the prediction;
5. **Compute the loss** (how wrong the prediction was);
6. **Backward**: compute how to adjust each weight (`loss.backward()`);
7. **Update** the weights (`optimizer.step()`).

Every few epochs, we store the train and validation MAE to later draw the
**loss curve** (and see whether it is learning / whether there is overfitting).

> It may take from seconds to a few minutes, depending on your hardware. On CPU, that's fine —
> just wait. If you want to go faster to test, lower `epochs`.


In [ ]:
epochs = 500            # how many times we go through the whole training set
record_every = 10       # how often (in epochs) we store the metrics

history = {'epoch': [], 'mae_train': [], 'mae_val': []}

for epoch in range(epochs):
    # ---- one full pass over the training set, batch by batch ----
    for X, y in train_loader:
        y = target_scaler.scale(y)                 # 1) scale the target
        X = X.to(compute_device, dtype=data_type)  # 2) data -> device
        y = y.to(compute_device, dtype=data_type)

        optimizer.zero_grad()                      # 3) zero the gradients
        output = model.forward(X).flatten()        # 4) forward (prediction)
        loss = criterion(output.view(-1), y.view(-1))  # 5) compute the loss
        loss.backward()                            # 6) backward (gradients)
        optimizer.step()                           # 7) update the weights

    # ---- record progress from time to time ----
    if epoch % record_every == 0 or epoch == epochs - 1:
        act_trn, pred_trn = predict(model, train_loader)   # performance on training
        act_val, pred_val = predict(model, val_loader)     # performance on validation
        mae_train = mean_absolute_error(act_trn, pred_trn)
        mae_val   = mean_absolute_error(act_val, pred_val)
        history['epoch'].append(epoch)
        history['mae_train'].append(mae_train)
        history['mae_val'].append(mae_val)
        print(f'epoch {epoch:3d} | train MAE: {mae_train:6.2f} | val MAE: {mae_val:6.2f}')

print('\nTraining finished!')

epoch   0 | train MAE:  31.42 | val MAE:  36.87
epoch  10 | train MAE:  10.56 | val MAE:  17.18
epoch  20 | train MAE:   9.12 | val MAE:  15.54
epoch  30 | train MAE:   8.41 | val MAE:  15.15
epoch  40 | train MAE:   8.02 | val MAE:  15.08
epoch  50 | train MAE:   7.78 | val MAE:  14.94
epoch  60 | train MAE:   7.56 | val MAE:  15.28
epoch  70 | train MAE:   7.47 | val MAE:  15.42
epoch  80 | train MAE:   7.31 | val MAE:  15.72
epoch  90 | train MAE:   7.24 | val MAE:  15.58
epoch 100 | train MAE:   7.23 | val MAE:  15.08
epoch 110 | train MAE:   6.97 | val MAE:  15.22
epoch 120 | train MAE:   6.44 | val MAE:  15.56
epoch 130 | train MAE:   6.07 | val MAE:  15.30
epoch 140 | train MAE:   5.71 | val MAE:  16.00
epoch 150 | train MAE:   5.40 | val MAE:  15.91
epoch 160 | train MAE:   5.26 | val MAE:  15.68
epoch 170 | train MAE:   5.15 | val MAE:  15.63
epoch 180 | train MAE:   5.02 | val MAE:  15.25
epoch 190 | train MAE:   4.91 | val MAE:  15.57
epoch 200 | train MAE:   4.87 | val MAE:

## Step 12 — Loss curve (training diagnostic)

The **loss curve** shows the MAE dropping over the epochs, on training and validation.
What to look for:
- both curves **going down** → the network is learning. 👍
- the validation curve **going up** while the training one keeps dropping →
  a sign of **overfitting** (the network started to memorize the training set).


In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(history['epoch'], history['mae_train'], '-o', ms=4, label='train')
plt.plot(history['epoch'], history['mae_val'],   '-s', ms=4, label='validation')
plt.xlabel('Epoch')
plt.ylabel('MAE in $C_p$ (J / mol K)')
plt.title('Loss curve')
plt.legend()
plt.show()

## Step 13 — Evaluate on validation

Now we measure the final performance on **validation** and draw the predicted × actual plot.


In [ ]:
act_val, pred_val = predict(model, val_loader)
r2, mae, rmse = evaluate(act_val, pred_val)
print(f'Validation -> R2: {r2:0.4f} | MAE: {mae:0.4f} | RMSE: {rmse:0.4f}')

plot_pred_act(act_val, pred_val, model)
plt.show()

## Step 14 — Evaluate on test (only once!)

The **test** set simulates "never-seen data". The golden rule:
> **only look at the test once**, when the model is already finalized.
If you use the test to make decisions and tune the model, it stops being
an honest estimate of performance.


In [ ]:
act_test, pred_test = predict(model, test_loader)
r2, mae, rmse = evaluate(act_test, pred_test)
print(f'Test -> R2: {r2:0.4f} | MAE: {mae:0.4f} | RMSE: {rmse:0.4f}')

plot_pred_act(act_test, pred_test, model)
plt.show()

## Comparing metric and diagnostics:

In [ ]:
def diagnose(name, loader):
    act, pred = predict(model, loader)
    act, pred = np.asarray(act), np.asarray(pred)
    r2   = r2_score(act, pred)
    mae  = mean_absolute_error(act, pred)
    rmse = np.sqrt(mean_squared_error(act, pred))
    print(f'{name:5s} | n={len(act):5d} | '
          f'std(Cp)={act.std():7.2f} | range=[{act.min():6.1f}, {act.max():6.1f}] | '
          f'R2={r2:6.3f} | MAE={mae:6.2f} | RMSE={rmse:6.2f}')

print(f'{"set":5s} | {"n":>5s} | {"std(Cp)":>11s} | {"range":>16s} | {"R2":>6s} | {"MAE":>6s} | {"RMSE":>6s}')
print('-'*82)
diagnose('train', train_loader)
diagnose('valid', val_loader)
diagnose('test',  test_loader)

## Step 15 — Save and load the model (checkpoint)

After training, we save the network **weights** (and the target scaler) to a
`.pth` file. This way you can reuse the model later **without training again** — and other
people can **reproduce** your work.

We store two things in the checkpoint:
1. `model.state_dict()` → all the weights and biases of the network;
2. `target_scaler.state_dict()` → to be able to unscale the predictions later.


In [ ]:
# --- Save ---
save_dict = {
    'weights': model.state_dict(),
    'scaler_state': target_scaler.state_dict(),
}
pth_path = 'model_checkpoint.pth'
torch.save(save_dict, pth_path)
print(f'Model saved to: {pth_path}')

In [ ]:
# --- Load (recreating the model from scratch and injecting the weights) ---
# 1) Recreate the SAME architecture
loaded_model = DenseNet(input_dims, hidden_dims=[16], dropout=0.0).to(compute_device)
loaded_scaler = MeanLogNormScaler(torch.ones(2))  # temporary values

# 2) Read the checkpoint and inject the weights
checkpoint = torch.load(pth_path, map_location=compute_device)
loaded_model.load_state_dict(checkpoint['weights'])
loaded_scaler.load_state_dict(checkpoint['scaler_state'])

# 3) Check that the loaded model gives the SAME result on the test set
#    (we need to point the global target_scaler to the loaded one, since 'predict' uses it)
target_scaler = loaded_scaler
act_test, pred_test = predict(loaded_model, test_loader)
r2, mae, rmse = evaluate(act_test, pred_test)
print(f'Loaded model -> R2: {r2:0.4f} | MAE: {mae:0.4f} | RMSE: {rmse:0.4f}')
print('If the numbers match Step 14, save/load worked!')

## Conclusion and next steps

You have just trained, evaluated and saved a **neural network** to predict $C_p$ — the same
problem as the classical part, now with deep learning. The flow is always the same:

**data → features → scaling → model → training → evaluation → save.**

### Exercises to play with (change one thing at a time!)
1. **Deeper network:** replace `hidden_dims=[16]` with `[64, 32]`. Does $R^2$ improve?
2. **Learning rate:** try `lr=1e-3` and `lr=1e-1`. What changes in the loss curve?
3. **Dropout:** use `dropout=0.2` when creating the model. Does it help against overfitting?
4. **More epochs:** increase `epochs` to 1000. Does validation keep improving or plateau?
5. **Batch size:** change `batch_size` (Step 7) from 64 to 256, and then to 16.
   Smaller batches add noise and take more update steps per epoch; larger batches are
   smoother and faster per epoch. Does the loss curve or the validation MAE change?
6. **Compare with the classical model:** does the best `scikit-learn` model (previous part) win
   or lose to the neural network on this test set?

> Best-practice tip: for a **fair** comparison, use exactly the same
> `cp_train/val/test.csv` on both sides (classical and DNN). You already did this here. ✅
